In [1]:
import sys
print(sys.executable)

/Users/mdtahsinsharif/anaconda3/envs/stream-metrics/bin/python


In [2]:
import os
import tarfile
import shutil
import glob
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path


In [3]:
# ===================== CONFIG =====================
query = "q7"
failure = "scalability"
cores_list = ["p4", "p10", "p20", "p30"]
#base_dir = Path(failure + "_flink")
base_dir = Path.cwd()
output_dir = Path("analysis_output") / failure / query
window_seconds = 5
# ==================================================

In [4]:
output_dir.mkdir(parents=True, exist_ok=True)

print("[START] Analysis pipeline starting...")
print(f"Query: {query}")
print(f"Base dir: {base_dir}")
print(f"Output dir: {output_dir}")

pattern = f"nexmark_{failure}_10p_{query}_*"
run_dirs = sorted([p for p in base_dir.glob(pattern) if p.is_dir()])

print(f"[INFO] Found {len(run_dirs)} run directories")

if not run_dirs:
    print(f"No runs found for {query}")
    exit(1)

node_data = {}
throughput_summary = {}  # {node: {cores: throughput}}

[START] Analysis pipeline starting...
Query: q7
Base dir: /Users/mdtahsinsharif/Desktop/Tahsin/Tahsin/tahsinTransferred/tahsinNewLaptop/studies/ece1724/project/metrics/scalability_flink
Output dir: analysis_output/scalability/q7
[INFO] Found 2 run directories


In [5]:
# ===================== HELPERS =====================
def compute_latency_summary(df):
    return {
        "avg": df["latency_ms"].mean(),
        "p50": df["latency_ms"].quantile(0.50),
        "p95": df["latency_ms"].quantile(0.95),
        "p99": df["latency_ms"].quantile(0.99),
    }


In [6]:
def compute_latency(df):
    r = df["latency_ms"].resample(f"{window_seconds}s")
    return pd.DataFrame({
        "p50": r.quantile(0.5),
        "p95": r.quantile(0.95),
        "p99": r.quantile(0.99),
    })

def compute_throughput(df):
    return df["auction"].resample(f"{window_seconds}s").size()

def smooth(series, w=5):
    return series.rolling(w, min_periods=1).mean()

def detect_failures(tp):
    gaps = []
    start = None

    for ts, v in tp.items():
        if v == 0 and start is None:
            start = ts
        elif v > 0 and start is not None:
            gaps.append((start, ts))
            start = None

    return gaps

In [7]:
def parse_part_file(pf):
    rows = []
    bad = 0

    print(f"[FILE] Processing: {os.path.basename(pf)}")

    with open(pf, "r", errors="ignore") as f:
        for line_num, line in enumerate(f, 1):
            parts = line.rstrip("\n").split(",")

            if len(parts) < 4:
                bad += 1
                continue

            auction, price, event_time, processed_time = parts

            try:
                rows.append({
                    "auction": int(auction),
                    "price": float(price),
                    "event_time": event_time,
                    "processed_time": processed_time
                })
            except:
                bad += 1

    print(f"    -> parsed rows: {len(rows)} | bad rows: {bad}")
    return pd.DataFrame(rows)

In [8]:
# ===================== MAIN LOOP =====================
for run_dir in run_dirs:
    tar_name = run_dir.name
    node = tar_name.split("_")[-1]

    print("\n========================================")
    print(f"[RUN] {tar_name}")
    print(f"[NODE] {node}")

    for cores in cores_list:
        data_path = run_dir / "nfs" / "flink" / "nexmark" / failure / query / cores

        print(f"[DATA] {cores} → {data_path}")

        if not data_path.exists():
            print(f"[WARN] Missing path")
            continue

        part_files = sorted(glob.glob(str(data_path / "part-*")))

        if not part_files:
            print(f"[WARN] No part files")
            continue

        dfs = []
        for pf in part_files:
            df_part = parse_part_file(pf)
            if not df_part.empty:
                dfs.append(df_part)

        if not dfs:
            continue

        df = pd.concat(dfs, ignore_index=True)

        # ---- timestamps ----
        df["event_time"] = pd.to_datetime(df["event_time"], errors="coerce")
        df["processed_time"] = pd.to_datetime(df["processed_time"], errors="coerce")

        df = df.dropna(subset=["event_time", "processed_time"])

        df["processed_time"] += pd.Timedelta(hours=4)

        # ---- latency ----
        df["latency_ms"] = (
            (df["processed_time"] - df["event_time"])
            .dt.total_seconds() * 1000
        )

        df = df.sort_values("processed_time").set_index("processed_time")

        # store only one core (for time-series plots)
        if cores == "p4":
            node_data[node] = df

        # ---- throughput summary ----
        tp = compute_throughput(df)

        # steady-state avg (ignore first 20%)
        tp_ss = tp[int(len(tp)*0.2):]
        avg_tp = tp_ss.mean() if not tp_ss.empty else tp.mean()

        throughput_summary.setdefault(node, {})[cores] = avg_tp

        print(f"[TP] {node} {cores}: {avg_tp:.2f}")

# ===================== LATENCY SUMMARY =====================
print("\n================ LATENCY SUMMARY ================\n")

for node, df in node_data.items():
    stats = compute_latency_summary(df)

    print(f"[{node}]")
    print(f"  Avg latency : {stats['avg']:.2f} ms")
    print(f"  P50 latency : {stats['p50']:.2f} ms")
    print(f"  P95 latency : {stats['p95']:.2f} ms")
    print(f"  P99 latency : {stats['p99']:.2f} ms")
    print()


[RUN] nexmark_scalability_10p_q7_n0
[NODE] n0
[DATA] p4 → /Users/mdtahsinsharif/Desktop/Tahsin/Tahsin/tahsinTransferred/tahsinNewLaptop/studies/ece1724/project/metrics/scalability_flink/nexmark_scalability_10p_q7_n0/nfs/flink/nexmark/scalability/q7/p4
[WARN] Missing path
[DATA] p10 → /Users/mdtahsinsharif/Desktop/Tahsin/Tahsin/tahsinTransferred/tahsinNewLaptop/studies/ece1724/project/metrics/scalability_flink/nexmark_scalability_10p_q7_n0/nfs/flink/nexmark/scalability/q7/p10
[WARN] Missing path
[DATA] p20 → /Users/mdtahsinsharif/Desktop/Tahsin/Tahsin/tahsinTransferred/tahsinNewLaptop/studies/ece1724/project/metrics/scalability_flink/nexmark_scalability_10p_q7_n0/nfs/flink/nexmark/scalability/q7/p20
[WARN] Missing path
[DATA] p30 → /Users/mdtahsinsharif/Desktop/Tahsin/Tahsin/tahsinTransferred/tahsinNewLaptop/studies/ece1724/project/metrics/scalability_flink/nexmark_scalability_10p_q7_n0/nfs/flink/nexmark/scalability/q7/p30
[WARN] No part files

[RUN] nexmark_scalability_10p_q7_n1
[NODE

In [9]:
# ===================== PLOTTING =====================
node_colors = {"n0": "blue", "n1": "orange"}

# ---- Latency ----
print("[PLOT] Latency...")
plt.figure()

for node, df in node_data.items():
    lat = compute_latency(df)

    plt.plot(smooth(lat["p50"]), label=f"{node} p50", color=node_colors.get(node))
    plt.plot(smooth(lat["p95"]), "--", label=f"{node} p95", color=node_colors.get(node))
    plt.plot(smooth(lat["p99"]), ":", label=f"{node} p99", color=node_colors.get(node))


[PLOT] Latency...


<Figure size 640x480 with 0 Axes>

In [10]:
plt.title(f"Latency ({query})")
plt.xlabel("Time")
plt.ylabel("ms")
plt.legend()
plt.tight_layout()
plt.savefig(output_dir / f"latency_{query}.png")
plt.close()

# ---- Throughput over time ----
print("[PLOT] Throughput (time)...")
plt.figure()

all_failures = []

for node, df in node_data.items():
    tp = compute_throughput(df)
    tp_s = smooth(tp)

    plt.plot(tp_s, label=node, color=node_colors.get(node))

    gaps = detect_failures(tp)
    for start, end in gaps:
        all_failures.append((start, end, node))

for start, end, node in all_failures:
    plt.axvspan(start, end, alpha=0.15, color=node_colors.get(node))

plt.title(f"Throughput ({query})")
plt.xlabel("Time")
plt.ylabel("events/sec")
plt.legend()
plt.tight_layout()
plt.savefig(output_dir / f"throughput_{query}.png")
plt.close()

# ---- Throughput vs cores ----
print("[PLOT] Throughput vs cores...")
plt.figure()

core_values = [int(c.replace("p", "")) for c in cores_list]

for node, core_map in throughput_summary.items():
    y = [core_map.get(c, 0) for c in cores_list]
    plt.plot(core_values, y, marker="o", label=node)

plt.title(f"Throughput vs Cores ({query})")
plt.xlabel("Number of Cores")
plt.ylabel("Events/sec")
plt.grid(True, linestyle="--", alpha=0.5)
plt.legend()
plt.tight_layout()

plt.savefig(output_dir / f"throughput_vs_cores_{query}.png")
plt.close()

print("[COMPLETE] Done. Outputs in:", output_dir)

[PLOT] Throughput (time)...
[PLOT] Throughput vs cores...
[COMPLETE] Done. Outputs in: analysis_output/scalability/q7


/var/folders/q5/_vyqqcz16vg3sg5pn1mmprcw0000gn/T/ipykernel_48571/3648427989.py:4: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  plt.legend()
/var/folders/q5/_vyqqcz16vg3sg5pn1mmprcw0000gn/T/ipykernel_48571/3648427989.py:31: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  plt.legend()
/var/folders/q5/_vyqqcz16vg3sg5pn1mmprcw0000gn/T/ipykernel_48571/3648427989.py:50: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  plt.legend()
